In [3]:
!pip install requests kaspa base58



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 11.4 MB/s eta 0:00:00


In [5]:
!pip install requests ecdsa



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.8/150.8 kB 4.3 MB/s eta 0:00:00


In [ ]:
import os
import time
import requests
import hashlib
import ecdsa

# Kaspa Bech32 karakter seti
CHARSET = "qpzry9x8gf2tvdw0s3jn54khce6mua7l"

def get_balance(address):
    """Kaspa API üzerinden bakiye kontrolü yapar."""
    try:
        url = f"https://kaspa.org{address}/balance"
        res = requests.get(url, timeout=3)
        if res.status_code == 200:
            return res.json().get("balance", 0) / 100_000_000
    except: return 0
    return 0

def generate_kaspa_address():
    """Kütüphane bağımlılığı olmadan adres üretir."""
    # 1. Private Key
    priv_key = os.urandom(32)
    # 2. Public Key (secp256k1)
    sk = ecdsa.SigningKey.from_string(priv_key, curve=ecdsa.SECP256k1)
    vk = sk.get_verifying_key()
    public_key = b'\x04' + vk.to_string() # Uncompressed

    # 3. Hash ve Bech32 Kodlama
    h = hashlib.sha256(public_key).digest()[:20]
    address_suffix = "".join([CHARSET[d % 32] for d in h])
    return priv_key.hex(), f"kaspa:q{address_suffix}"

def start_hunt():
    print("[+] Kaspa Piyango Modu Başlatıldı...")
    print("[!] Bakiye bulunursa JACKPOT.txt dosyasına kaydedilecek.\n")

    count = 0
    start_time = time.time()

    try:
        while True:
            priv, addr = generate_kaspa_address()

            # API'yi yormamak için her 200 denemede bir bakiye kontrolü
            if count % 200 == 0:
                balance = get_balance(addr)
                if balance > 0:
                    result = f"BULDUM! | Adres: {addr} | Bakiye: {balance} | Key: {priv}\n"
                    print(f"\n\n{result}")
                    with open("/content/JACKPOT.txt", "a") as f:
                        f.write(result)

            count += 1
            if count % 500 == 0:
                hiz = count / (time.time() - start_time)
                print(f"\rTaranan: {count} | Hız: {hiz:.2f} adr/sn | Güncel: {addr[:25]}...", end="")

    except KeyboardInterrupt:
        print("\n[!] İşlem durduruldu.")

if __name__ == "__main__":
    start_hunt()


[+] Kaspa Piyango Modu Başlatıldı...
[!] Bakiye bulunursa JACKPOT.txt dosyasına kaydedilecek.

Taranan: 27500 | Hız: 1110.71 adr/sn | Güncel: kaspa:qmjjy2kmady7l2zf0g2...